**BERN02 Exercise: Generalised Linear Regression**

Name: Yang Shann Wen

Date: 6 September 2026

In this exercise, we will use the bird count data to generate three samples of hypothetical data from 1999 to 2012. We formulate a **Poisson regression model** (suitable for response variables of 'counts')

$Y_i \sim \mathrm{Poisson}(\lambda_i)$

$\log(\lambda_i) = \beta_0 + \beta_1(\mathrm{year}_i - 1999)$

$\lambda_i = \exp(\beta_0 + \beta_1(\mathrm{year}_i - 1999))$

<br>

and estimate the parameters using **Maximum Likelihood Estimation**.

$logL(θ) = \sum_{i=1}^{n}[-\lambda_i + y_i\log(\lambda_i) - \log(y_i!)]$

$\hat{\theta} = \arg\max_{\theta} \log L(\theta)$




In [60]:
import pandas as pd
import numpy as np
from scipy.special import gammaln
from scipy.optimize import minimize

In [61]:
# Import and load dataset
df = pd.read_csv('/content/bird_count.csv')

In [62]:
# Extract the year (predictor x) and the bird count (response y) from their respective columns
x = df['yr']
y = df['count']

In [63]:
# Shift the year so that 1999 is the baseline year to prevent exponential overflow and improves numerical stability during optimization
# beta_0 represents the log expected bird count in 1999
x_baseline = 1999
x_diff = x - x_baseline

In [64]:
# We want to find parameters that maximize the log likelihood. Since scipy's
# minimize() looks for the smallest value, minimizing the negative log likelihood
# gives us the maximum likelihood estimates.

# Define the negative log likelihood function
def negative_log_likelihood(theta, x, y):
  """
  Parameters:
  theta: Parameters of the model (beta_0, beta_1)
  x: The predictor observations (year - 1999)
  y: The response variable observations (bird count)

  Output:
  -log_1: The value of negative log likelihood

  """

  beta_0, beta_1 = theta

  # Calculate the expected value of the response variable y (lambda) using a Poisson GLM:
  # log(lambda) = beta_0 + beta_1 * (year - 1999)
  # lambda = exp(beta_0 + beta_1 * (year - 1999))
  lam = np.exp(beta_0 + beta_1 * x)

  # Calculate the likelihood for Poisson GLM:
  # log likelihood = sum of (-lambda) + y * log(lambda) - log(y!)
  # We use gammaln(y+1) to calculate log(y!)
  log_factorial_y = gammaln(y + 1)
  log_l = np.sum(-lam + y * np.log(lam) - log_factorial_y)

  return -log_l

In [69]:
# Initiate parameter guesses
initial_guess = [0.0, 0.0]

# Run optimization to find the parameters that minimize the negative log likelihood
result = minimize(
    fun=negative_log_likelihood,
    x0=initial_guess,
    args=(x_diff, y),
    method="BFGS"
)

print("Optimization successful:", result.success)
print("Optimization message:", result.message)

# Extract beta_0_hat and beta_1_hat from result
beta_0_hat, beta_1_hat = result.x

print("\nEstimated parameters:")
print(f"beta_0_hat = {beta_0_hat}")
print(f"beta_1_hat = {beta_1_hat}")

Optimization successful: True
Optimization message: Optimization terminated successfully.

Estimated parameters:
beta_0_hat = 2.3253319407283524
beta_1_hat = -0.03244317973781568


In [66]:
# To generate the three hypothetical samples, we need to compute the point estimates for expected bird count (lambda_hat) for each year
lambda_hat = np.exp(beta_0_hat + beta_1_hat * x_diff)

In [72]:
# Generate the three hypothetical samples
# Set seed for reproducibility
np.random.seed(42)

# Initiate a blank list to store the sample data
sample_data = []
num_samples = 3

for sample_num in range(1, num_samples + 1):

    # Draw random counts from Poisson(lambda_hat)
    hypothetical_counts = np.random.poisson(lam=lambda_hat)

    for yr, expected_lam, hypo_cnt in zip(x.astype(int), lambda_hat, hypothetical_counts):
        sample_data.append({
            "Sample_number": sample_num,
            "Year": yr,
            "Hypothetical_bird_count": hypo_cnt,
            "Expected_lambda": np.round(expected_lam, 3)
        })

# Convert the sample data into a data frame
sample_df = pd.DataFrame(sample_data)

In [73]:
# Save sample to CSV file
output_filename = "bird_count_samples.csv"
sample_df.to_csv(output_filename, index=False)
print(f"Successfully generated 3 samples and saved to '{output_filename}'.")

# Preview the generated data
print("\nFirst 10 rows of generated samples:")
print(sample_df.head(10).to_string(index=False))

Successfully generated 3 samples and saved to 'bird_count_samples.csv'.

First 10 rows of generated samples:
 Sample_number  Year  Hypothetical_bird_count  Expected_lambda
             1  2011                        6            6.931
             1  2010                        7            7.160
             1  2002                        7            9.281
             1  2006                        7            8.152
             1  2008                        6            7.640
             1  2012                        4            6.710
             1  2000                       14            9.904
             1  2005                        6            8.421
             1  2007                        7            7.892
             1  2004                       10            8.698
